# Translation Quality LLM Scoring (blocked vs free)

This notebook compares two decoding settings for each model:
- **blocked**: translations generated with no-repeat-3-gram
- **free**: translations generated without n-gram blocking

For each model id, it expects two TSV files in `INPUT_DIR`:
- `{id}_blocked_translations.tsv`
- `{id}_free_translations.tsv`

The same source sentences are selected in both conditions and scored by the LLM, so that the difference comes only from the decoding strategy.

In [1]:
# 1. Install necessary libraries
!pip install transformers datasets accelerate bitsandbytes tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.0 MB/s eta 0:00:00


In [2]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# 3. Set Working Directory to thoth_eval
import os
%cd /content/drive/MyDrive/thoth_eval/

# Ensure outputs folder exists
os.makedirs("outputs", exist_ok=True)
!ls

/content/drive/MyDrive/thoth_eval
7_generation_eval.py	     model.py			  tokenizer
8_llm_scoring.py	     outputs			  translation_samples
llm_scoring_colab.ipynb      __pycache__
llm_scoring_colab_old.ipynb  thoth_generation_eval.ipynb


In [4]:
# 4. Scoring Pipeline — Ablation: blocked vs free decoding
import csv
import json
import random
import torch
import gc
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# --- Configuration ---
INPUT_DIR = "translation_samples"
OUTPUT_DIR = "outputs"
TOP_N = 10
SEED = 42

model_name = "Qwen/Qwen2.5-7B-Instruct"
scorer_key = "qwen"

SCORING_PROMPT = """You are an expert evaluator for English-to-French machine translation.

**Source (English):**
{en}

**Reference (French):**
{ref}

**Model Output (French):**
{gen}

Evaluate the model output on these criteria (1-5 scale):
1. **Accuracy**: Does it correctly convey the meaning of the source?
2. **Fluency**: Is the French grammatically correct and natural?
3. **Completeness**: Does it translate the full sentence without missing parts?
4. **Conciseness**: Does it avoid unnecessary repetition or extra content?

Respond ONLY with a JSON object:
{{"accuracy": X, "fluency": X, "completeness": X, "conciseness": X, "overall": X, "comment": "brief explanation"}}"""

# --- Helper Functions ---
def load_tsv(path):
    """Load TSV with columns EN, REF, GEN (plus optional gen_len/ref_len/rep_rate)."""
    samples = []
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            gen, ref = row["GEN"], row["REF"]
            samples.append({
                "en": row["EN"],
                "ref": ref,
                "gen": gen,
                "gen_len": int(row.get("gen_len", len(gen))),
                "ref_len": int(row.get("ref_len", len(ref))),
                "rep_rate": float(row.get("rep_rate", 0.0)),
            })
    return samples

def select_keys_from_common(blocked_by_en, free_by_en, top_n=10):
    """Stratified selection from sentences present in BOTH conditions.
    Groups: high_ratio (most over-generated), mid_ratio (closest to ref length), random."""
    rng = random.Random(SEED)
    common_ens = set(blocked_by_en.keys()) & set(free_by_en.keys())
    print(f"  Common source sentences: {len(common_ens)} / blocked={len(blocked_by_en)} / free={len(free_by_en)}")

    pool = []
    for en in common_ens:
        s = blocked_by_en[en]
        s["len_ratio"] = s["gen_len"] / max(1, s["ref_len"])
        pool.append(s)

    high = sorted(pool, key=lambda x: x["len_ratio"], reverse=True)[:top_n]
    mid = sorted(pool, key=lambda x: abs(x["len_ratio"] - 1.0))[:top_n]
    used_ens = {s["en"] for s in high} | {s["en"] for s in mid}
    rest = [s for s in pool if s["en"] not in used_ens]
    rand = rng.sample(rest, min(top_n, len(rest))) if rest else []

    selected = {}
    for group_name, group in [("high_ratio", high), ("mid_ratio", mid), ("random", rand)]:
        for s in group:
            if s["en"] not in selected:
                selected[s["en"]] = group_name
    return selected

def query_llm(model, tokenizer, en, ref, gen):
    prompt = SCORING_PROMPT.format(en=en, ref=ref, gen=gen)
    if hasattr(tokenizer, "apply_chat_template"):
        messages = [{"role": "user", "content": prompt}]
        formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    else:
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=200, do_sample=False, pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    try:
        start = response.find("{")
        end = response.rfind("}") + 1
        if start >= 0 and end > start:
            return json.loads(response[start:end])
    except:
        pass
    return {"error": "No JSON found", "raw": response[:200]}

def score_condition(samples_by_en, selected_keys, condition_tag, model, tokenizer):
    """Score selected samples for one decoding condition."""
    to_score = []
    for en_key, reason in selected_keys.items():
        if en_key in samples_by_en:
            s = samples_by_en[en_key]
            to_score.append({**s, "selection_reason": reason, "condition": condition_tag})

    scored = []
    for s in tqdm(to_score, desc=f"  {condition_tag}"):
        r = query_llm(model, tokenizer, s["en"], s["ref"], s["gen"])
        scored.append({**s, "llm_scores": r})
    return scored

def aggregate(scored):
    valid = [r for r in scored if "error" not in r["llm_scores"]]
    dims = ["accuracy", "fluency", "completeness", "conciseness", "overall"]
    if not valid:
        return {"error": "No valid scores"}, {}
    avgs = {d: round(sum(r["llm_scores"].get(d, 0) for r in valid) / len(valid), 2) for d in dims}
    by_reason = {}
    for r in valid:
        gk = r["selection_reason"]
        by_reason.setdefault(gk, []).append(r["llm_scores"])
    reason_avgs = {
        gk: {"count": len(v), "avg_overall": round(sum(s.get("overall", 0) for s in v) / len(v), 2)}
        for gk, v in by_reason.items()
    }
    return avgs, reason_avgs

# --- Discover Model IDs & build global intersection ---
BLOCKED_SUFFIX = "_blocked_translations.tsv"
FREE_SUFFIX = "_free_translations.tsv"

blocked_files = {f.replace(BLOCKED_SUFFIX, ""): f for f in os.listdir(INPUT_DIR) if f.endswith(BLOCKED_SUFFIX)}
free_files = {f.replace(FREE_SUFFIX, ""): f for f in os.listdir(INPUT_DIR)
              if f.endswith(FREE_SUFFIX) and not f.endswith(BLOCKED_SUFFIX) and "_beam" not in f}
model_ids = sorted(set(blocked_files.keys()) & set(free_files.keys()))
print(f"Found {len(model_ids)} models with BOTH conditions: {model_ids}")

# Load all TSVs and compute global EN intersection
all_blocked = {}
all_free = {}
global_ens = None
for mid in model_ids:
    b = load_tsv(os.path.join(INPUT_DIR, blocked_files[mid]))
    f = load_tsv(os.path.join(INPUT_DIR, free_files[mid]))
    b_by_en = {s["en"]: s for s in b}
    f_by_en = {s["en"]: s for s in f}
    all_blocked[mid] = b_by_en
    all_free[mid] = f_by_en
    common = set(b_by_en.keys()) & set(f_by_en.keys())
    global_ens = common if global_ens is None else global_ens & common

print(f"Global intersection (same sentences across ALL models x 2 conditions): {len(global_ens)}")

# Stratified selection from global intersection (using first model's blocked data for len_ratio)
first_blocked = all_blocked[model_ids[0]]
global_pool = [first_blocked[en] for en in global_ens]
selected_keys = select_keys_from_common(
    {en: first_blocked[en] for en in global_ens},
    {en: all_free[model_ids[0]][en] for en in global_ens},
    top_n=TOP_N,
)
print(f"Selected {len(selected_keys)} sentences for ALL models (high_ratio / mid_ratio / random, {TOP_N} each)")

print(f"\n{'='*60}")
print(f"LOADING SCORER: {scorer_key} ({model_name})")
print(f"{'='*60}")

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
qconfig = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(
    model_name, quantization_config=qconfig, device_map="auto", trust_remote_code=True
)
model.eval()

all_summaries = []

for model_id in model_ids:
    print(f"\n>>> {model_id}: scoring blocked + free")

    blocked_by_en = all_blocked[model_id]
    free_by_en = all_free[model_id]

    scored_blocked = score_condition(blocked_by_en, selected_keys, "blocked", model, tokenizer)
    scored_free = score_condition(free_by_en, selected_keys, "free", model, tokenizer)

    avgs_b, reason_b = aggregate(scored_blocked)
    avgs_f, reason_f = aggregate(scored_free)

    dims = ["accuracy", "fluency", "completeness", "conciseness", "overall"]
    delta = {}
    if isinstance(avgs_b, dict) and isinstance(avgs_f, dict) and "error" not in avgs_b and "error" not in avgs_f:
        delta = {d: round(avgs_b[d] - avgs_f[d], 2) for d in dims}

    summary = {
        "model_id": model_id,
        "num_paired": len(selected_keys),
        "blocked": {"averages": avgs_b, "by_selection": reason_b},
        "free": {"averages": avgs_f, "by_selection": reason_f},
        "delta_blocked_minus_free": delta,
    }
    all_summaries.append(summary)

    out_base = os.path.join(OUTPUT_DIR, f"{model_id}_llm_ablation_{scorer_key}")
    with open(out_base + ".json", "w", encoding="utf-8") as f:
        json.dump({"summary": summary, "blocked": scored_blocked, "free": scored_free}, f, indent=2, ensure_ascii=False)

    lines = [
        f"{'='*55}",
        f"LLM ABLATION: {model_id} ({scorer_key})",
        f"{'='*55}",
        f"Paired samples: {len(selected_keys)}",
        "",
        "--- BLOCKED (no_repeat_ngram_size=3) ---",
    ]
    if "error" not in avgs_b:
        for d in dims:
            lines.append(f"  {d.capitalize():<15} | {avgs_b[d]:>5.2f}")
    lines.append("\n--- FREE (original greedy) ---")
    if "error" not in avgs_f:
        for d in dims:
            lines.append(f"  {d.capitalize():<15} | {avgs_f[d]:>5.2f}")
    if delta:
        lines.append("\n--- DELTA (blocked - free) ---")
        for d in dims:
            lines.append(f"  {d.capitalize():<15} | {delta[d]:>+5.2f}")
    with open(out_base + ".txt", "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print(f"  -> {out_base}.json & .txt saved")

# --- Final cross-model ablation summary ---
print(f"\n{'='*60}")
print("CROSS-MODEL ABLATION SUMMARY")
print(f"{'='*60}")
header = f"{'Model':<12} | {'Blocked':>8} | {'Free':>8} | {'Delta':>8}"
print(header)
print("-" * len(header))
for s in all_summaries:
    b = s["blocked"]["averages"].get("overall", "N/A")
    f_ = s["free"]["averages"].get("overall", "N/A")
    d = s["delta_blocked_minus_free"].get("overall", "N/A")
    b_s = f"{b:>8.2f}" if isinstance(b, (int, float)) else f"{b:>8}"
    f_s = f"{f_:>8.2f}" if isinstance(f_, (int, float)) else f"{f_:>8}"
    d_s = f"{d:>+8.2f}" if isinstance(d, (int, float)) else f"{d:>8}"
    print(f"{s['model_id']:<12} | {b_s} | {f_s} | {d_s}")

abl_path = os.path.join(OUTPUT_DIR, f"llm_ablation_summary_{scorer_key}.json")
with open(abl_path, "w", encoding="utf-8") as f:
    json.dump(all_summaries, f, indent=2, ensure_ascii=False)
print(f"\nSummary saved -> {abl_path}")


Found 6 models with BOTH conditions: ['r0v0', 'r0v1', 'r0v2', 'r0v3', 'r0v4', 'r0v5']
Global intersection (same sentences across ALL models x 2 conditions): 257
  Common source sentences: 257 / blocked=257 / free=257
Selected 30 sentences for ALL models (high_ratio / mid_ratio / random, 10 each)

LOADING SCORER: qwen (Qwen/Qwen2.5-7B-Instruct)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

  free:   0%|          | 0/30 [00:00<?, ?it/s]

  -> outputs/r0v4_llm_ablation_qwen.json & .txt saved

>>> r0v5: scoring blocked + free


  blocked:   0%|          | 0/30 [00:00<?, ?it/s]

  free:   0%|          | 0/30 [00:00<?, ?it/s]

  -> outputs/r0v5_llm_ablation_qwen.json & .txt saved

CROSS-MODEL ABLATION SUMMARY
Model        |  Blocked |     Free |    Delta
---------------------------------------------
r0v0         |     2.68 |     3.36 |    -0.68
r0v1         |     3.09 |     3.50 |    -0.41
r0v2         |     2.78 |     3.38 |    -0.60
r0v3         |     2.85 |     3.36 |    -0.51
r0v4         |     3.07 |     3.62 |    -0.55
r0v5         |     3.05 |     3.49 |    -0.44

Summary saved -> outputs/llm_ablation_summary_qwen.json
